# Colab PPO 500 dòng cho Qwen2.5-VL sau SFT/QLoRA

Notebook này dùng các thư mục có sẵn trên Google Drive, chỉ fine-tune PPO trên **500 dòng đầu tiên** từ `PPO/ppo_train_5000.jsonl`. Checkpoint lưu trực tiếp vào Drive nên Colab bị ngắt có thể chạy lại để resume.


## 1. Cài thư viện


In [1]:
# ============================================================
# 1. CÀI THƯ VIỆN
# ============================================================
!pip -q install -U "transformers>=4.45.0" "accelerate>=0.33.0" "peft>=0.12.0" bitsandbytes qwen-vl-utils pillow tqdm pandas safetensors
!pip -q install -U bert-score


## 2. Import + kiểm tra GPU


In [2]:
# ============================================================
# 2. IMPORT + KIỂM TRA GPU
# ============================================================
import os, re, gc, json, math, time, shutil, random, string, zipfile
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from tqdm.auto import tqdm

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('BF16 supported:', torch.cuda.is_bf16_supported())


CUDA: True
GPU: Tesla T4
BF16 supported: True


## 3. Mount Drive + cấu hình


In [3]:
# ============================================================
# 3. MOUNT DRIVE + CONFIG COLAB
# Chỉ fine-tune PPO trên 500 dòng đầu tiên.
# Resume thật vì checkpoint nằm trong Google Drive.
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

FINAL_DL_ROOT = Path('/content/drive/MyDrive/Final_Deeplearning')

# File PPO JSONL đã có sẵn trên Drive
PPO_JSONL = FINAL_DL_ROOT / 'PPO' / 'ppo_train_5000.jsonl'

# Adapter SFT/final adapter đã train trước đó
SFT_FINAL_ADAPTER = FINAL_DL_ROOT / 'qwen25_vl_herb_qlora_10chunks' / 'final_finetuned_adapter'

# Output PPO 500 dòng. Lưu trong Drive để Colab ngắt vẫn resume được.
PPO_RUN_ROOT = FINAL_DL_ROOT / 'PPO' / 'qwen25_vl_ppo_colab_500_vqa_softacc_norm_bertscore_resume'
PPO_RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Nếu có images_final.zip thì notebook sẽ unzip vào Drive một lần.
IMAGES_ZIP = FINAL_DL_ROOT / 'images_final.zip'
UNZIP_IMAGE_ROOT = FINAL_DL_ROOT / '_images_final_unzipped'

# Root tìm ảnh. Có thể thêm folder ảnh khác vào list này nếu cần.
IMAGE_SEARCH_ROOTS = [
    UNZIP_IMAGE_ROOT,
    FINAL_DL_ROOT,
    FINAL_DL_ROOT / 'Split',
    FINAL_DL_ROOT / 'images',
    FINAL_DL_ROOT / 'PPO',
]

# Model + train config
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
MAX_TRAIN_ROWS = 1000
CHUNK_SIZE = 1000
TOTAL_ROWS_EXPECTED = 1000
RUN_CHUNK_IDS = [1]

# PPO config nhỏ, an toàn hơn trên Colab
LR = 5e-6
PPO_EPOCHS = 1
CLIP_EPS = 0.2
VALUE_COEF = 0.1
KL_COEF = 0.05
MAX_NEW_TOKENS = 32

LOG_EVERY_STEPS = 5
SAVE_EVERY_STEPS = 25

# Reward config
REWARD_ALPHA = 0.5
REWARD_BETA = 0.5
BERTSCORE_MODEL = 'xlm-roberta-base'
BERTSCORE_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BERTSCORE_BATCH_SIZE = 8
BERTSCORE_NORM_FLOOR = 0.80

print('FINAL_DL_ROOT     :', FINAL_DL_ROOT)
print('PPO_JSONL         :', PPO_JSONL, PPO_JSONL.exists())
print('SFT_FINAL_ADAPTER :', SFT_FINAL_ADAPTER, SFT_FINAL_ADAPTER.exists())
print('PPO_RUN_ROOT      :', PPO_RUN_ROOT)
print('MAX_TRAIN_ROWS    :', MAX_TRAIN_ROWS)

if not PPO_JSONL.exists():
    raise FileNotFoundError(f'Không thấy file PPO JSONL: {PPO_JSONL}')
if not SFT_FINAL_ADAPTER.exists():
    raise FileNotFoundError(f'Không thấy final adapter: {SFT_FINAL_ADAPTER}')

if IMAGES_ZIP.exists():
    marker = UNZIP_IMAGE_ROOT / '.unzipped_done'
    UNZIP_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
    if not marker.exists():
        print('Unzipping images:', IMAGES_ZIP)
        with zipfile.ZipFile(IMAGES_ZIP, 'r') as z:
            z.extractall(UNZIP_IMAGE_ROOT)
        marker.write_text('done', encoding='utf-8')
        print('Unzip done:', UNZIP_IMAGE_ROOT)
    else:
        print('Images already unzipped:', UNZIP_IMAGE_ROOT)
else:
    print('Không thấy images_final.zip. Notebook sẽ tìm ảnh trực tiếp trong Drive.')


Mounted at /content/drive
FINAL_DL_ROOT     : /content/drive/MyDrive/Final_Deeplearning
PPO_JSONL         : /content/drive/MyDrive/Final_Deeplearning/PPO/ppo_train_5000.jsonl True
SFT_FINAL_ADAPTER : /content/drive/MyDrive/Final_Deeplearning/qwen25_vl_herb_qlora_10chunks/final_finetuned_adapter True
PPO_RUN_ROOT      : /content/drive/MyDrive/Final_Deeplearning/PPO/qwen25_vl_ppo_colab_500_vqa_softacc_norm_bertscore_resume
MAX_TRAIN_ROWS    : 1000
Images already unzipped: /content/drive/MyDrive/Final_Deeplearning/_images_final_unzipped


## 4. Đọc PPO JSONL + lấy 1000 dòng


In [4]:
# ============================================================
# 4. ĐỌC PPO JSONL + LẤY 500 DÒNG ĐẦU
# ============================================================
def read_jsonl_safe(path):
    rows, bad = [], []
    with open(path, 'r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception as e:
                bad.append((line_no, repr(e), line[:300]))
    if bad:
        print(f'[WARNING] {path.name}: {len(bad)} dòng JSON lỗi. In 5 dòng đầu:')
        for x in bad[:5]:
            print(x)
    return rows

ppo_rows_all = read_jsonl_safe(PPO_JSONL)
ppo_rows = ppo_rows_all[:MAX_TRAIN_ROWS]
print('Total PPO rows in file:', len(ppo_rows_all))
print('PPO rows used:', len(ppo_rows))

CHUNKS = [{
    'chunk_id': 1,
    'start': 0,
    'end': len(ppo_rows),
    'name': f'chunk_01_00001_{len(ppo_rows):05d}',
}]

print('Chunks:')
for c in CHUNKS:
    print(c)
assert len(ppo_rows) > 0, 'Không có dòng PPO nào để train.'


Total PPO rows in file: 5000
PPO rows used: 1000
Chunks:
{'chunk_id': 1, 'start': 0, 'end': 1000, 'name': 'chunk_01_00001_01000'}


## 5. Xử lý path ảnh + reference + vqa_soft_acc


In [5]:
IMAGE_EXTS = ['.jpg', '.jpeg', '.png', '.webp', '.bmp']
_IMAGE_BASENAME_CACHE = {}


def get_first_existing_key(row, keys):
    for k in keys:
        v = row.get(k)
        if v is not None and str(v).strip():
            return str(v).strip()
    return ''


def _search_by_basename(name):
    """Fallback tìm ảnh theo basename trong /kaggle/input. Có cache để tránh rglob lặp lại."""
    name = str(name)
    if name in _IMAGE_BASENAME_CACHE:
        return _IMAGE_BASENAME_CACHE[name]

    matches = []
    for root in IMAGE_SEARCH_ROOTS:
        root = Path(root)
        if not root.exists():
            continue
        try:
            found = list(root.rglob(name))
            matches.extend([p for p in found if p.is_file()])
        except Exception:
            pass

    _IMAGE_BASENAME_CACHE[name] = matches[0] if matches else None
    return _IMAGE_BASENAME_CACHE[name]


def resolve_image_path(row):
    raw = get_first_existing_key(row, ['image', 'image_path', 'path', 'file_name', 'filename'])
    if not raw:
        raise ValueError(f'Row thiếu trường image/image_path/path/file_name: {row}')

    # Nếu JSONL còn path Drive/Colab, Kaggle không có path đó. Lấy basename để tìm lại trong /kaggle/input.
    p = Path(raw)
    if p.is_absolute() and p.exists():
        return p

    basename = p.name
    candidates = []
    for root in IMAGE_SEARCH_ROOTS:
        root = Path(root)
        candidates.extend([
            root / raw,
            root / 'images' / raw,
            root / basename,
            root / 'images' / basename,
        ])

    if p.suffix == '':
        for root in IMAGE_SEARCH_ROOTS:
            root = Path(root)
            for ext in IMAGE_EXTS:
                candidates.append(root / f'{raw}{ext}')
                candidates.append(root / 'images' / f'{raw}{ext}')
                candidates.append(root / f'{basename}{ext}')
                candidates.append(root / 'images' / f'{basename}{ext}')

    for c in candidates:
        if c.exists() and c.is_file():
            return c

    found = _search_by_basename(basename)
    if found is not None:
        return found

    raise FileNotFoundError(
        'Không tìm thấy ảnh cho raw=' + raw + '\n'
        'Hãy kiểm tra lại ảnh trên Google Drive hoặc sửa IMAGE_SEARCH_ROOTS.\n'
        'Ví dụ path đã thử:\n' + '\n'.join(str(x) for x in candidates[:20])
    )


def get_question(row):
    return get_first_existing_key(row, ['question', 'query', 'prompt'])


def _as_answer_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if isinstance(value, tuple):
        return [str(x).strip() for x in value if str(x).strip()]
    return [str(value).strip()] if str(value).strip() else []


def get_reference_answers(row):
    """
    Trả về list reference answers để tính vqa_soft_acc.
    - Nếu dataset có nhiều đáp án/aliases thì dùng tất cả.
    - Nếu chỉ có 1 reference_answer/answer thì list có 1 phần tử.
    """
    multi_keys = [
        'reference_answers', 'references', 'answers', 'ground_truths',
        'gt_answers', 'labels', 'aliases', 'acceptable_answers'
    ]
    single_keys = ['reference_answer', 'answer', 'ground_truth', 'gt', 'label']

    refs = []
    for key in multi_keys:
        if key in row:
            refs.extend(_as_answer_list(row.get(key)))

    for key in single_keys:
        if key in row:
            refs.extend(_as_answer_list(row.get(key)))

    # Khử trùng lặp nhưng giữ thứ tự.
    seen = set()
    clean_refs = []
    for x in refs:
        if x not in seen:
            clean_refs.append(x)
            seen.add(x)
    return clean_refs


def get_reference_answer(row):
    refs = get_reference_answers(row)
    return refs[0] if refs else ''


VI_PUNCT = string.punctuation + '“”‘’…–—。、，！？：；（）[]{}'


def normalize_vqa_answer(text):
    """
    Normalize answer cho VQA tiếng Việt:
    - lowercase
    - bỏ dấu câu
    - gom khoảng trắng
    - bỏ vài tiền tố thường gặp trong đáp án ngắn: cây, màu, dược liệu...
    """
    if text is None:
        return ''

    text = str(text).lower().strip()
    text = re.sub(r'[{}]'.format(re.escape(VI_PUNCT)), ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    prefixes = [
        'cây ',
        'loài cây ',
        'dược liệu ',
        'vị thuốc ',
        'màu ',
    ]

    changed = True
    while changed:
        changed = False
        for prefix in prefixes:
            if text.startswith(prefix):
                text = text[len(prefix):].strip()
                changed = True

    return text


def answer_match(pred, ref):
    p = normalize_vqa_answer(pred)
    r = normalize_vqa_answer(ref)
    if not p or not r:
        return False
    return p == r


def vqa_soft_acc(pred, refs):
    """
    VQA soft accuracy đúng logic VQA:
        score = min(match_count / 3, 1.0)

    Nếu chỉ có 1 reference:
        match -> 1.0
        không match -> 0.0

    Nếu có nhiều reference/aliases:
        match_count = số reference khớp prediction sau normalize
        score = min(match_count / 3, 1.0)
    """
    if refs is None:
        return 0.0
    if isinstance(refs, str):
        refs = [refs]
    if not isinstance(refs, list):
        refs = [str(refs)]

    refs = [str(x).strip() for x in refs if str(x).strip()]
    if len(refs) == 0:
        return 0.0

    match_count = sum(1 for ref in refs if answer_match(pred, ref))

    if len(refs) == 1:
        return 1.0 if match_count >= 1 else 0.0

    return min(match_count / 3.0, 1.0)


for i, row in enumerate(ppo_rows[:3], start=1):
    print('Row', i)
    print('  question:', get_question(row)[:100])
    print('  refs    :', get_reference_answers(row)[:5])
    print('  image   :', resolve_image_path(row))


Row 1
  question: Thực thể trong hình được sử dụng làm vị thuốc gì?
  refs    : ['Cây hẹ']
  image   : /content/drive/MyDrive/Final_Deeplearning/_images_final_unzipped/images_final/P000002266.jpg
Row 2
  question: Hoa trong ảnh có màu gì?
  refs    : ['Màu trắng']
  image   : /content/drive/MyDrive/Final_Deeplearning/_images_final_unzipped/images_final/P000002266.jpg
Row 3
  question: Các nhị hoa trong hình có màu gì?
  refs    : ['Màu vàng']
  image   : /content/drive/MyDrive/Final_Deeplearning/_images_final_unzipped/images_final/P000002266.jpg


## 6. Load Qwen2.5-VL + LoRA adapter


In [6]:
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel
from qwen_vl_utils import process_vision_info

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = 'right'

def infer_qwen_vl_hidden_size(model):
    cfg = getattr(model, 'config', None)
    if cfg is None and hasattr(model, 'base_model'):
        cfg = getattr(model.base_model, 'config', None)
    for attr in ['hidden_size', 'd_model', 'n_embd']:
        if cfg is not None and hasattr(cfg, attr):
            return int(getattr(cfg, attr))
    if cfg is not None and hasattr(cfg, 'text_config'):
        text_cfg = cfg.text_config
        for attr in ['hidden_size', 'd_model', 'n_embd']:
            if hasattr(text_cfg, attr):
                return int(getattr(text_cfg, attr))
    raise AttributeError('Không tìm được hidden_size trong config/text_config.')

def sanitize_lora_adapter_dir(adapter_dir, work_dir):
    adapter_dir = Path(adapter_dir)
    work_dir = Path(work_dir)
    safe_dir = work_dir / '_safe_adapter_for_loading'
    if safe_dir.exists():
        shutil.rmtree(safe_dir)
    shutil.copytree(adapter_dir, safe_dir)
    cfg_path = safe_dir / 'adapter_config.json'
    cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
    # Chỉ match language layers, tránh visual.blocks gây missing adapter keys.
    cfg['target_modules'] = r'.*model\.layers\.\d+\.(self_attn\.(q_proj|k_proj|v_proj|o_proj)|mlp\.(gate_proj|up_proj|down_proj))$'
    cfg_path.write_text(json.dumps(cfg, ensure_ascii=False, indent=2), encoding='utf-8')
    return safe_dir

def load_policy_with_ref_adapter(source_adapter_dir, work_dir):
    source_adapter_dir = Path(source_adapter_dir)
    work_dir = Path(work_dir)
    safe_adapter_dir = sanitize_lora_adapter_dir(source_adapter_dir, work_dir)

    print('Loading base model:', MODEL_ID)
    base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
    )
    base_model.config.use_cache = False

    print('Loading policy adapter:', safe_adapter_dir)
    model = PeftModel.from_pretrained(
        base_model,
        str(safe_adapter_dir),
        adapter_name='default',
        is_trainable=True,
    )
    print('Loading ref adapter:', safe_adapter_dir)
    model.load_adapter(str(safe_adapter_dir), adapter_name='ref', is_trainable=False)

    model.set_adapter('default')
    model.train()
    model.print_trainable_parameters()
    return model


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## 7. Prompt, generate, logprob/value


In [7]:
def get_model_device(model):
    for p in model.parameters():
        return p.device
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def move_to_device(batch, device):
    out = {}
    for k, v in batch.items():
        out[k] = v.to(device) if torch.is_tensor(v) else v
    return out

def build_user_text(question):
    return (
        'Bạn là hệ thống hỏi đáp ảnh dược liệu Việt Nam. '
        'Hãy trả lời câu hỏi bằng tiếng Việt, thật ngắn gọn, tối đa 10 từ. '
        'Không giải thích thêm.\n'
        f'Câu hỏi: {question}'
    )

def make_prompt_messages(row):
    image_path = resolve_image_path(row)
    question = get_question(row)
    if not question:
        raise ValueError(f'Row thiếu question: {row}')
    return [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': str(image_path)},
            {'type': 'text', 'text': build_user_text(question)},
        ],
    }]

def encode_prompt(row, model):
    messages = make_prompt_messages(row)
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info([messages])
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors='pt')
    return move_to_device(inputs, get_model_device(model))

def generate_response(model, row, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    model.set_adapter('default')
    encoded = encode_prompt(row, model)
    prompt_len = int(encoded['input_ids'].shape[1])
    with torch.no_grad():
        generated_ids = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )
    response_ids = generated_ids[:, prompt_len:]
    pred = processor.tokenizer.batch_decode(response_ids, skip_special_tokens=True)[0].strip()
    return encoded, generated_ids, pred, prompt_len

def get_model_inputs_for_generated(encoded_inputs, generated_ids):
    out = {'input_ids': generated_ids, 'attention_mask': torch.ones_like(generated_ids)}
    for k in ['pixel_values', 'image_grid_thw', 'pixel_values_videos', 'video_grid_thw']:
        if k in encoded_inputs:
            out[k] = encoded_inputs[k]
    return out

def sequence_logprob_and_value(model, value_head, model_inputs, prompt_len):
    kwargs = dict(model_inputs)
    kwargs['output_hidden_states'] = True
    kwargs['return_dict'] = True
    out = model(**kwargs)

    logits = out.logits[:, :-1, :].float()
    labels = kwargs['input_ids'][:, 1:]
    logprobs = F.log_softmax(logits, dim=-1)
    token_logprobs = logprobs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)

    response_start = max(prompt_len - 1, 0)
    mask = torch.zeros_like(token_logprobs, dtype=torch.float32)
    mask[:, response_start:] = 1.0
    if processor.tokenizer.pad_token_id is not None:
        mask = mask * (labels != processor.tokenizer.pad_token_id).float()

    seq_logprob = (token_logprobs * mask).sum(dim=1)

    hidden_states = out.hidden_states[-1]
    seq_len = kwargs['input_ids'].shape[1]
    last_positions = []
    for b in range(kwargs['input_ids'].shape[0]):
        response_positions = torch.where(mask[b] > 0)[0] + 1
        pos = int(response_positions[-1].item()) if len(response_positions) else seq_len - 1
        last_positions.append(max(0, min(pos, seq_len - 1)))
    last_hidden = torch.stack([hidden_states[b, pos] for b, pos in enumerate(last_positions)], dim=0)

    vh_param = next(value_head.parameters())
    last_hidden = last_hidden.to(device=vh_param.device, dtype=vh_param.dtype)
    value = value_head(last_hidden).squeeze(-1).float()
    return seq_logprob.float(), value


## 8. Reward = 0.5 * vqa_soft_acc + 0.5 * normalized_BERTScore_F1


In [8]:
# ============================================================
# 8. REWARD ĐÚNG YÊU CẦU
# reward = 0.5 * vqa_soft_acc + 0.5 * normalized_BERTScore F1
# ============================================================

from bert_score import BERTScorer

print('Khởi tạo BERTScorer một lần để không load lại model mỗi step...')
BERT_SCORER = BERTScorer(
    model_type=BERTSCORE_MODEL,
    lang='vi',
    device=BERTSCORE_DEVICE,
    rescale_with_baseline=False,
)
print('BERTScorer ready.')
print('Reward formula: reward = 0.5 * vqa_soft_acc + 0.5 * normalized_BERTScore_F1')
print('BERTScore raw floor:', BERTSCORE_NORM_FLOOR)


def ref_to_string_for_bertscore(ref):
    """
    BERTScore cần reference dạng string.
    Nếu ref là list nhiều đáp án thì lấy đáp án đầu tiên làm reference chính.
    vqa_soft_acc vẫn dùng toàn bộ list refs.
    """
    if ref is None:
        return ''
    if isinstance(ref, list):
        return str(ref[0]).strip() if len(ref) > 0 else ''
    return str(ref).strip()


def compute_bertscore_raw_f1(preds, refs):
    """Tính BERTScore F1 raw bằng thư viện bert-score."""
    clean_preds = [str(p).strip() if p is not None else '' for p in preds]
    clean_refs = [ref_to_string_for_bertscore(r) for r in refs]

    with torch.no_grad():
        _, _, f1 = BERT_SCORER.score(
            clean_preds,
            clean_refs,
            batch_size=BERTSCORE_BATCH_SIZE,
            verbose=False,
        )

    return [float(x) for x in f1.detach().cpu().tolist()]


def normalize_bertscore_f1(raw_f1, floor=BERTSCORE_NORM_FLOOR):
    """
    Chuẩn hóa BERTScore F1 raw về thang hữu ích hơn cho reward.

    Lý do: BERTScore raw thường cao sẵn. Với câu tiếng Việt ngắn,
    prediction sai vẫn có thể đạt 0.75-0.85. Nếu cộng raw trực tiếp vào reward,
    câu sai vẫn được thưởng cao.

    Công thức:
        raw <= floor -> 0
        raw = 1       -> 1
        còn lại       -> (raw - floor) / (1 - floor)
    """
    x = float(raw_f1)
    if x <= floor:
        return 0.0
    return max(0.0, min(1.0, (x - floor) / max(1.0 - floor, 1e-8)))


def compute_rewards(preds, refs):
    """
    PPO reward:
        reward = 0.5 * vqa_soft_acc + 0.5 * normalized_BERTScore_F1

    Input:
        preds: list[str]
        refs : list[str] hoặc list[list[str]]

    Output:
        rewards: list[float]
        vqa_soft_accs: list[float]
        bertscore_f1_norms: list[float]
        bertscore_f1_raws: list[float]
    """
    assert len(preds) == len(refs), 'preds và refs phải cùng độ dài.'

    vqa_soft_accs = [vqa_soft_acc(pred, ref) for pred, ref in zip(preds, refs)]
    bertscore_f1_raws = compute_bertscore_raw_f1(preds, refs)
    bertscore_f1_norms = [normalize_bertscore_f1(x) for x in bertscore_f1_raws]

    rewards = []
    for vqa_s, bert_norm in zip(vqa_soft_accs, bertscore_f1_norms):
        reward = REWARD_ALPHA * float(vqa_s) + REWARD_BETA * float(bert_norm)
        reward = max(0.0, min(1.0, reward))
        rewards.append(reward)

    return rewards, vqa_soft_accs, bertscore_f1_norms, bertscore_f1_raws


# Test nhanh để nhìn đúng thành phần reward.
_test_preds = ['Cây rau má', 'Màu trắng', 'Cam thảo', 'Không rõ']
_test_refs = ['Rau má', 'Trắng', 'Rau má', ['Không rõ', 'không xác định', 'không biết']]
_test_rewards, _test_vqa, _test_bert_norm, _test_bert_raw = compute_rewards(_test_preds, _test_refs)

for p, r, rw, vqa_s, bert_norm, bert_raw in zip(_test_preds, _test_refs, _test_rewards, _test_vqa, _test_bert_norm, _test_bert_raw):
    print('-' * 80)
    print('Prediction        :', p)
    print('Reference         :', r)
    print('vqa_soft_acc      :', vqa_s)
    print('BERTScore F1 raw  :', bert_raw)
    print('BERTScore F1 norm :', bert_norm)
    print('reward            :', rw)


Khởi tạo BERTScorer một lần để không load lại model mỗi step...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScorer ready.
Reward formula: reward = 0.5 * vqa_soft_acc + 0.5 * normalized_BERTScore_F1
BERTScore raw floor: 0.8
--------------------------------------------------------------------------------
Prediction        : Cây rau má
Reference         : Rau má
vqa_soft_acc      : 1.0
BERTScore F1 raw  : 0.8886096477508545
BERTScore F1 norm : 0.44304823875427235
reward            : 0.7215241193771362
--------------------------------------------------------------------------------
Prediction        : Màu trắng
Reference         : Trắng
vqa_soft_acc      : 1.0
BERTScore F1 raw  : 0.7878904342651367
BERTScore F1 norm : 0.0
reward            : 0.5
--------------------------------------------------------------------------------
Prediction        : Cam thảo
Reference         : Rau má
vqa_soft_acc      : 0.0
BERTScore F1 raw  : 0.8689473271369934
BERTScore F1 norm : 0.3447366356849669
reward            : 0.17236831784248344
-------------------------------------------------------------------------

## 9. Save/resume trên Drive


In [9]:
# ============================================================
# 9. SAVE / RESUME STATE TRÊN GOOGLE DRIVE
# ============================================================
def save_value_head(value_head, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(value_head.state_dict(), path)


def load_value_head(value_head, path, device):
    path = Path(path)
    if path.exists():
        value_head.load_state_dict(torch.load(path, map_location=device))
        print('Loaded value_head:', path)
    return value_head


def save_optimizer(optimizer, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(optimizer.state_dict(), path)


def load_optimizer(optimizer, path, device):
    path = Path(path)
    if path.exists():
        optimizer.load_state_dict(torch.load(path, map_location=device))
        print('Loaded optimizer:', path)
    return optimizer


def read_state(state_path):
    state_path = Path(state_path)
    if state_path.exists():
        return json.loads(state_path.read_text(encoding='utf-8'))
    return None


def write_state(state, state_path):
    state_path = Path(state_path)
    state_path.parent.mkdir(parents=True, exist_ok=True)
    state_path.write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding='utf-8')


def export_policy_adapter(model, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    model.set_adapter('default')
    model.save_pretrained(str(out_dir), selected_adapters=['default'])
    processor.save_pretrained(str(out_dir))
    return out_dir

# No-op để train_one_chunk không còn phụ thuộc Hugging Face upload.
def upload_chunk_work_checkpoint_to_hf(*args, **kwargs):
    return False

def upload_chunk_adapter_to_hf(*args, **kwargs):
    return False


## 10. Train PPO 1 chunk 1000 dòng


In [10]:
def train_one_chunk(chunk):
    chunk_id = int(chunk['chunk_id'])
    chunk_name = chunk['name']
    start, end = int(chunk['start']), int(chunk['end'])
    chunk_rows = ppo_rows[start:end]

    work_dir = PPO_RUN_ROOT / f'{chunk_name}_work'
    adapter_dir = PPO_RUN_ROOT / f'{chunk_name}_adapter'
    latest_ckpt_dir = work_dir / 'latest_checkpoint'
    state_path = work_dir / 'ppo_state.json'
    log_path = work_dir / 'ppo_history.jsonl'
    work_dir.mkdir(parents=True, exist_ok=True)

    if chunk_id == 1:
        source_adapter = SFT_FINAL_ADAPTER
    else:
        prev_name = CHUNKS[chunk_id - 2]['name']
        source_adapter = PPO_RUN_ROOT / f'{prev_name}_adapter'
        if not source_adapter.exists():
            raise FileNotFoundError(f'Chunk {chunk_id} cần adapter chunk trước nhưng không thấy: {source_adapter}')

    state = read_state(state_path)
    if state and state.get('status') == 'done' and ((adapter_dir / 'adapter_model.safetensors').exists() or (adapter_dir / 'adapter_model.bin').exists()):
        print(f'[SKIP] {chunk_name} đã done:', adapter_dir)
        return adapter_dir

    resume_from_ckpt = latest_ckpt_dir.exists() and (latest_ckpt_dir / 'adapter_config.json').exists()
    load_adapter = latest_ckpt_dir if resume_from_ckpt else source_adapter
    print(('[RESUME]' if resume_from_ckpt else '[START]'), chunk_name, 'adapter=', load_adapter)

    model = load_policy_with_ref_adapter(load_adapter, work_dir)
    device = get_model_device(model)

    hidden_size = infer_qwen_vl_hidden_size(model)
    print('Value head hidden_size:', hidden_size)
    value_head = nn.Linear(hidden_size, 1).to(device=device, dtype=torch.float32)
    if resume_from_ckpt:
        load_value_head(value_head, latest_ckpt_dir / 'value_head.pt', device)

    trainable_params = list(filter(lambda p: p.requires_grad, model.parameters())) + list(value_head.parameters())
    optimizer = torch.optim.AdamW(trainable_params, lr=LR)
    if resume_from_ckpt:
        load_optimizer(optimizer, latest_ckpt_dir / 'optimizer.pt', device)

    start_step = int(state.get('next_step', 0)) if state else 0
    print('next_step:', start_step, '/', len(chunk_rows))
    if start_step == 0 and log_path.exists() and not resume_from_ckpt:
        log_path.unlink()

    history = []

    for local_step in tqdm(range(start_step, len(chunk_rows)), desc=f'PPO {chunk_name}'):
        row = chunk_rows[local_step]
        ref_answers = get_reference_answers(row)
        ref_answer = ref_answers[0] if ref_answers else ''
        if not ref_answers:
            print('[WARNING] skip missing refs at local_step=', local_step)
            continue

        # Generate
        model.set_adapter('default')
        encoded, generated_ids, pred, prompt_len = generate_response(model, row, MAX_NEW_TOKENS)
        model_inputs = get_model_inputs_for_generated(encoded, generated_ids)

        # Reward đúng: 0.5 * vqa_soft_acc + 0.5 * normalized_BERTScore_F1
        rewards, vqa_soft_accs, bertscore_f1_norms, bertscore_f1_raws = compute_rewards([pred], [ref_answers])
        reward = torch.tensor(rewards, dtype=torch.float32, device=device)

        # Old logprob/value
        model.eval(); model.set_adapter('default')
        with torch.no_grad():
            old_logp, old_value = sequence_logprob_and_value(model, value_head, model_inputs, prompt_len)

        # Ref logprob
        model.eval(); model.set_adapter('ref')
        with torch.no_grad():
            ref_logp, _ = sequence_logprob_and_value(model, value_head, model_inputs, prompt_len)

        # Update policy
        model.train(); model.set_adapter('default'); value_head.train()
        advantage = (reward - old_value.detach()).float()
        old_logp_detached = old_logp.detach()
        ref_logp_detached = ref_logp.detach()

        last_loss = last_policy_loss = last_value_loss = last_kl_loss = None
        for _ in range(PPO_EPOCHS):
            optimizer.zero_grad(set_to_none=True)
            new_logp, value = sequence_logprob_and_value(model, value_head, model_inputs, prompt_len)
            log_ratio = (new_logp - old_logp_detached).clamp(-10, 10)
            ratio = torch.exp(log_ratio)
            unclipped = ratio * advantage
            clipped = torch.clamp(ratio, 1.0 - CLIP_EPS, 1.0 + CLIP_EPS) * advantage
            policy_loss = -torch.min(unclipped, clipped).mean()
            value_loss = F.mse_loss(value.float(), reward.float())
            kl_loss = ((new_logp.float() - ref_logp_detached.float()) ** 2).mean()
            loss = policy_loss + VALUE_COEF * value_loss + KL_COEF * kl_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            last_loss = float(loss.detach().cpu())
            last_policy_loss = float(policy_loss.detach().cpu())
            last_value_loss = float(value_loss.detach().cpu())
            last_kl_loss = float(kl_loss.detach().cpu())

        rec = {
            'global_step': local_step + 1,
            'chunk_id': chunk_id,
            'chunk_name': chunk_name,
            'row_index_0_based': start + local_step,
            'id': row.get('id'),
            'question': get_question(row),
            'reference_answer': ref_answer,
            'prediction': pred,
            'reward': float(reward.detach().cpu()[0]),
            'vqa_soft_acc': float(vqa_soft_accs[0]),
            'bertscore_f1_norm': float(bertscore_f1_norms[0]),
            'bertscore_f1_raw': float(bertscore_f1_raws[0]),
            'old_logp': float(old_logp.detach().cpu()[0]),
            'ref_logp': float(ref_logp.detach().cpu()[0]),
            'loss': last_loss,
            'policy_loss': last_policy_loss,
            'value_loss': last_value_loss,
            'kl_loss': last_kl_loss,
            'time': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
        }
        history.append(rec)
        with open(log_path, 'a', encoding='utf-8') as f:
            f.write(json.dumps(rec, ensure_ascii=False) + '\n')

        if (local_step + 1) % LOG_EVERY_STEPS == 0:
            recent = history[-LOG_EVERY_STEPS:]
            mean_reward = sum(x['reward'] for x in recent) / len(recent)
            mean_vqa_soft = sum(x['vqa_soft_acc'] for x in recent) / len(recent)
            mean_bert_norm = sum(x['bertscore_f1_norm'] for x in recent) / len(recent)
            mean_bert_raw = sum(x['bertscore_f1_raw'] for x in recent) / len(recent)
            print(
                f'[{chunk_name} step {local_step+1}/{len(chunk_rows)}] '
                f'reward={mean_reward:.4f} '
                f'vqa_soft_acc={mean_vqa_soft:.4f} '
                f'bertscore_norm={mean_bert_norm:.4f} '
                f'bertscore_raw={mean_bert_raw:.4f} '
                f'loss={last_loss:.4f}'
            )

        if ((local_step + 1) % SAVE_EVERY_STEPS == 0) or ((local_step + 1) == len(chunk_rows)):
            if latest_ckpt_dir.exists():
                shutil.rmtree(latest_ckpt_dir)
            export_policy_adapter(model, latest_ckpt_dir)
            save_value_head(value_head, latest_ckpt_dir / 'value_head.pt')
            save_optimizer(optimizer, latest_ckpt_dir / 'optimizer.pt')
            write_state({
                'status': 'running',
                'chunk_id': chunk_id,
                'chunk_name': chunk_name,
                'start': start,
                'end': end,
                'next_step': local_step + 1,
                'source_adapter': str(source_adapter),
                'latest_checkpoint': str(latest_ckpt_dir),
                'updated_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
            }, state_path)
            print(f'[SAVE] {chunk_name}: checkpoint step {local_step + 1}')
            # Colab/Drive version: checkpoint đã nằm trong Drive, không upload HF.

        if torch.cuda.is_available() and ((local_step + 1) % 10 == 0):
            torch.cuda.empty_cache()

    # Export adapter cuối chunk
    if adapter_dir.exists():
        shutil.rmtree(adapter_dir)
    export_policy_adapter(model, adapter_dir)
    save_value_head(value_head, adapter_dir / 'value_head.pt')

    all_logs = read_jsonl_safe(log_path) if log_path.exists() else []
    rewards = [x.get('reward', 0.0) for x in all_logs]
    vqa_softs = [x.get('vqa_soft_acc', 0.0) for x in all_logs]
    bert_norms = [x.get('bertscore_f1_norm', 0.0) for x in all_logs]
    bert_raws = [x.get('bertscore_f1_raw', 0.0) for x in all_logs]
    summary = {
        'train_mode': 'ppo_after_sft_2chunks_resume',
        'chunk_id': chunk_id,
        'chunk_name': chunk_name,
        'source_adapter': str(source_adapter),
        'adapter_dir': str(adapter_dir),
        'work_dir': str(work_dir),
        'num_rows': len(chunk_rows),
        'start_index_0_based': start,
        'end_index_0_based_exclusive': end,
        'lr': LR,
        'ppo_epochs': PPO_EPOCHS,
        'clip_eps': CLIP_EPS,
        'value_coef': VALUE_COEF,
        'kl_coef': KL_COEF,
        'max_new_tokens': MAX_NEW_TOKENS,
        'reward_formula': '0.5 * vqa_soft_acc + 0.5 * normalized_BERTScore_F1',
        'reward_alpha_vqa_soft_acc': REWARD_ALPHA,
        'reward_beta_normalized_bertscore_f1': REWARD_BETA,
        'bertscore_norm_floor': BERTSCORE_NORM_FLOOR,
        'bertscore_model': BERTSCORE_MODEL,
        'mean_reward': sum(rewards) / len(rewards) if rewards else None,
        'mean_vqa_soft_acc': sum(vqa_softs) / len(vqa_softs) if vqa_softs else None,
        'mean_bertscore_f1_norm': sum(bert_norms) / len(bert_norms) if bert_norms else None,
        'mean_bertscore_f1_raw': sum(bert_raws) / len(bert_raws) if bert_raws else None,
        'finished_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    (adapter_dir / 'ppo_chunk_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    write_state({**summary, 'status': 'done', 'next_step': len(chunk_rows), 'latest_checkpoint': str(latest_ckpt_dir)}, state_path)

    # Colab/Drive version: adapter/work_dir đã nằm trong Drive.

    del model, value_head, optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f'[DONE] {chunk_name}:', adapter_dir)
    return adapter_dir


## 11. Chạy training


In [11]:
# ============================================================
# 11. CHẠY PPO 1000 DÒNG
# Nếu Colab bị ngắt, chạy lại notebook từ đầu: nó sẽ resume từ Drive.
# ============================================================
completed_adapters = []
for chunk in CHUNKS:
    if chunk['chunk_id'] not in RUN_CHUNK_IDS:
        continue
    adapter_dir = train_one_chunk(chunk)
    completed_adapters.append(str(adapter_dir))

print('Completed adapters:')
for p in completed_adapters:
    print(' -', p)


[RESUME] chunk_01_00001_01000 adapter= /content/drive/MyDrive/Final_Deeplearning/PPO/qwen25_vl_ppo_colab_500_vqa_softacc_norm_bertscore_resume/chunk_01_00001_01000_work/latest_checkpoint
Loading base model: Qwen/Qwen2.5-VL-3B-Instruct


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Loading policy adapter: /content/drive/MyDrive/Final_Deeplearning/PPO/qwen25_vl_ppo_colab_500_vqa_softacc_norm_bertscore_resume/chunk_01_00001_01000_work/_safe_adapter_for_loading
Loading ref adapter: /content/drive/MyDrive/Final_Deeplearning/PPO/qwen25_vl_ppo_colab_500_vqa_softacc_norm_bertscore_resume/chunk_01_00001_01000_work/_safe_adapter_for_loading
trainable params: 14,966,784 || all params: 3,784,556,544 || trainable%: 0.3955
Value head hidden_size: 2048
Loaded value_head: /content/drive/MyDrive/Final_Deeplearning/PPO/qwen25_vl_ppo_colab_500_vqa_softacc_norm_bertscore_resume/chunk_01_00001_01000_work/latest_checkpoint/value_head.pt
Loaded optimizer: /content/drive/MyDrive/Final_Deeplearning/PPO/qwen25_vl_ppo_colab_500_vqa_softacc_norm_bertscore_resume/chunk_01_00001_01000_work/latest_checkpoint/optimizer.pt
next_step: 975 / 1000


PPO chunk_01_00001_01000:   0%|          | 0/25 [00:00<?, ?it/s]

[chunk_01_00001_01000 step 980/1000] reward=0.2836 vqa_soft_acc=0.2000 bertscore_norm=0.3672 bertscore_raw=0.8723 loss=-0.3637
[chunk_01_00001_01000 step 985/1000] reward=0.3177 vqa_soft_acc=0.2000 bertscore_norm=0.4355 bertscore_raw=0.8837 loss=0.2863
[chunk_01_00001_01000 step 990/1000] reward=0.1870 vqa_soft_acc=0.0000 bertscore_norm=0.3739 bertscore_raw=0.8693 loss=0.4155
[chunk_01_00001_01000 step 995/1000] reward=0.3615 vqa_soft_acc=0.2000 bertscore_norm=0.5230 bertscore_raw=0.9046 loss=-0.2874
[chunk_01_00001_01000 step 1000/1000] reward=0.3517 vqa_soft_acc=0.2000 bertscore_norm=0.5034 bertscore_raw=0.8992 loss=-0.5792
[SAVE] chunk_01_00001_01000: checkpoint step 1000
[DONE] chunk_01_00001_01000: /content/drive/MyDrive/Final_Deeplearning/PPO/qwen25_vl_ppo_colab_500_vqa_softacc_norm_bertscore_resume/chunk_01_00001_01000_adapter
Completed adapters:
 - /content/drive/MyDrive/Final_Deeplearning/PPO/qwen25_vl_ppo_colab_500_vqa_softacc_norm_bertscore_resume/chunk_01_00001_01000_adapte

## 12. Export final adapter


In [12]:
# ============================================================
# 12. EXPORT FINAL PPO ADAPTER + ZIP VỀ DRIVE
# ============================================================
final_source = None
for chunk in reversed(CHUNKS):
    p = PPO_RUN_ROOT / f"{chunk['name']}_adapter"
    if p.exists() and ((p / 'adapter_model.safetensors').exists() or (p / 'adapter_model.bin').exists()):
        final_source = p
        break
if final_source is None:
    raise FileNotFoundError('Chưa tìm thấy adapter PPO chunk nào để export final.')

FINAL_PPO_ADAPTER = PPO_RUN_ROOT / 'final_ppo_adapter'
FINAL_PPO_ZIP = PPO_RUN_ROOT / 'final_ppo_adapter.zip'
if FINAL_PPO_ADAPTER.exists():
    shutil.rmtree(FINAL_PPO_ADAPTER)
shutil.copytree(final_source, FINAL_PPO_ADAPTER)

summary = {
    'export_type': 'final_ppo_lora_adapter_colab_500_rows',
    'source_adapter_dir': str(final_source),
    'final_adapter_dir': str(FINAL_PPO_ADAPTER),
    'final_zip_path': str(FINAL_PPO_ZIP),
    'base_model': MODEL_ID,
    'sft_start_adapter': str(SFT_FINAL_ADAPTER),
    'ppo_jsonl': str(PPO_JSONL),
    'num_train_rows': len(ppo_rows),
    'reward_formula': '0.5 * vqa_soft_acc + 0.5 * normalized_BERTScore_F1',
    'bertscore_norm_floor': BERTSCORE_NORM_FLOOR,
    'created_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
(FINAL_PPO_ADAPTER / 'final_ppo_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

if FINAL_PPO_ZIP.exists():
    FINAL_PPO_ZIP.unlink()
shutil.make_archive(str(FINAL_PPO_ZIP).replace('.zip', ''), 'zip', FINAL_PPO_ADAPTER)
print('FINAL_PPO_ADAPTER:', FINAL_PPO_ADAPTER)
print('FINAL_PPO_ZIP    :', FINAL_PPO_ZIP)


FINAL_PPO_ADAPTER: /content/drive/MyDrive/Final_Deeplearning/PPO/qwen25_vl_ppo_colab_500_vqa_softacc_norm_bertscore_resume/final_ppo_adapter
FINAL_PPO_ZIP    : /content/drive/MyDrive/Final_Deeplearning/PPO/qwen25_vl_ppo_colab_500_vqa_softacc_norm_bertscore_resume/final_ppo_adapter.zip


## 13. Xem log


In [13]:
# ============================================================
# 13. XEM LOG PPO
# ============================================================
all_logs = []
for chunk in CHUNKS:
    log_path = PPO_RUN_ROOT / f"{chunk['name']}_work" / 'ppo_history.jsonl'
    if log_path.exists():
        all_logs.extend(read_jsonl_safe(log_path))

log_df = pd.DataFrame(all_logs)
print('Total logs:', len(log_df))
if len(log_df):
    display(log_df.tail(20))
    for col in ['reward', 'vqa_soft_acc', 'bertscore_f1_norm', 'bertscore_f1_raw', 'loss']:
        if col in log_df:
            print(f'Mean {col}:', log_df[col].mean())
else:
    print('Chưa có log.')


Total logs: 1005


,global_step,chunk_id,chunk_name,row_index_0_based,id,question,reference_answer,prediction,reward,vqa_soft_acc,bertscore_f1_norm,bertscore_f1_raw,old_logp,ref_logp,loss,policy_loss,value_loss,kl_loss,time
985,981,1,chunk_01_00001_01000,980,ppo_000981,Lá của cây So đũa có hình dạng như thế nào?,Lá của cây So đũa có dạng lá kép lông,Lá mọc đối xứng,0.000000,0.0,0.000000,0.790418,-8.324960,-8.130575,0.231352,0.228739,0.025915,4.178367e-04,2026-05-08T03:10:30Z
986,982,1,chunk_01_00001_01000,981,ppo_000982,Có bao nhiêu bông hoa lớn đang nở rõ ràng ở tr...,Có,Có,1.000000,1.0,1.000000,1.000000,-0.064912,-0.064017,0.026106,0.025641,0.004645,1.214425e-07,2026-05-08T03:10:36Z
987,983,1,chunk_01_00001_01000,982,ppo_000983,Các nụ hoa của cây So đũa có màu gì và hình dạ...,Màu xanh lục và hình dạng thuôn dài,"Màu xanh lục, hình bầu dục",0.297451,0.0,0.594902,0.918980,-4.123562,-4.078835,0.139321,0.138013,0.012126,1.912627e-03,2026-05-08T03:10:44Z
988,984,1,chunk_01_00001_01000,983,ppo_000984,Vị trí của các nụ hoa nằm ở đâu so với các bôn...,Các nụ hoa nằm ở phía trên và xung quanh,Các nụ hoa nằm ở dưới cùng của cụm hoa,0.291233,0.0,0.582465,0.916493,-7.220393,-7.344176,-0.216661,-0.230036,0.125983,1.553334e-02,2026-05-08T03:10:53Z
989,985,1,chunk_01_00001_01000,984,ppo_000985,Bề mặt của các bông hoa So đũa trong ảnh trông...,Màu trắng sáng,Bề mặt của các bông hoa So đũa trông rất bóng,0.000000,0.0,0.000000,0.792393,-6.230597,-6.260394,0.286313,0.281163,0.051408,1.810347e-04,2026-05-08T03:11:02Z
990,986,1,chunk_01_00001_01000,985,ppo_000986,Cây So đũa trong ảnh có đang ra quả không?,Không,Có,0.488336,0.0,0.976673,0.995335,-0.143269,-0.132771,0.821923,0.761315,0.605781,5.894647e-04,2026-05-08T03:11:08Z
991,987,1,chunk_01_00001_01000,986,ppo_000987,Tên dược liệu trong ảnh là gì?,Cây So đũa,Đậu bắp,0.000000,0.0,0.000000,0.781283,-5.872797,-5.880790,0.237704,0.226773,0.108732,1.166231e-03,2026-05-08T03:11:15Z
992,988,1,chunk_01_00001_01000,987,ppo_000988,Dược liệu So đũa có công dụng gì theo thông ti...,Chữa tưa lưỡi,Chữa rắn cắn,0.140237,0.0,0.280474,0.856095,-5.735049,-5.741253,-0.054297,-0.055062,0.000305,1.469339e-02,2026-05-08T03:11:22Z
993,989,1,chunk_01_00001_01000,988,ppo_000989,Bộ phận nào của cây So đũa nổi bật nhất trong ...,Bộ phận nổi bật nhất trong ảnh là những bông,Bộ phận nổi bật nhất là các bông hoa màu đỏ,0.306204,0.0,0.612408,0.922482,-4.846509,-5.040788,-0.226737,-0.232520,0.031579,5.248560e-02,2026-05-08T03:11:30Z
994,990,1,chunk_01_00001_01000,989,ppo_000990,Hoa của cây So đũa trong ảnh có màu gì?,Hoa của cây So đũa có màu đỏ thẫm kết,Màu đỏ hồng,0.000000,0.0,0.000000,0.791228,-1.923228,-1.889605,0.415515,0.399440,0.159926,1.645752e-03,2026-05-08T03:11:37Z


Mean reward: 0.25160352738935554
Mean vqa_soft_acc: 0.14527363184079603
Mean bertscore_f1_norm: 0.35793342258206645
Mean bertscore_f1_raw: 0.8656931395554424
Mean loss: 0.06458297076505326
